<a href="https://colab.research.google.com/github/minjikim0330/ml_project/blob/main/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
import gradio as gr

# =====================================
# ⭐ 머신러닝 학습 데이터
# =====================================

data = {
    "temp": [0, 3, 5, 8, 10, 13, 16, 20, 23, 26, 30],

    "feel_temp": [-5, -1, 2, 6, 8, 11, 15, 19, 24, 28, 33],

    # 맑음=0, 비=1, 눈=2
    "weather": [2, 2, 1, 0, 0, 1, 0, 0, 0, 1, 0],

    # 머신러닝이 예측할 상의
    "top": [
        "패딩",
        "코트",
        "니트",
        "맨투맨",
        "얇은 니트",
        "가디건",
        "얇은 긴팔",
        "긴팔티",
        "반팔",
        "반팔",
        "민소매"
    ]
}

df = pd.DataFrame(data)

# 입력값(X)
X = df[["temp", "feel_temp", "weather"]]

# 정답(y)
y = df["top"]

# =====================================
# ⭐ Decision Tree 모델 생성 및 학습
# =====================================

model = DecisionTreeClassifier()

model.fit(X, y)

# =====================================
# ⭐ 하의 데이터
# =====================================

male_bottoms = {
    "패딩": ["기모바지", "두꺼운 바지"],
    "코트": ["청바지", "슬랙스"],
    "니트": ["청바지", "면바지"],
    "맨투맨": ["청바지", "면바지"],
    "얇은 니트": ["슬랙스", "청바지"],
    "가디건": ["면바지", "청바지"],
    "얇은 긴팔": ["청바지", "면바지"],
    "긴팔티": ["면바지", "반바지"],
    "반팔": ["반바지", "면바지"],
    "민소매": ["반바지"]
}

female_bottoms = {
    "패딩": ["기모바지", "기모 치마", "기모 청바지"],
    "코트": ["치마", "청바지", "롱스커트", "트레이닝 바지", "슬렉스"],
    "니트": ["치마", "슬랙스"],
    "맨투맨": ["청바지", "치마"],
    "얇은 니트": ["슬랙스", "치마"],
    "가디건": ["치마", "청바지"],
    "얇은 긴팔": ["청바지", "치마"],
    "긴팔티": ["치마", "반바지"],
    "반팔": ["반바지", "치마"],
    "민소매": ["치마", "반바지"]
}

# =====================================
# ⭐ 이너웨어 데이터
# =====================================

inners = {
    "패딩": "히트텍",
    "코트": "히트텍",
    "니트": "히트텍",
    "맨투맨": "없음",
    "얇은 니트": "없음",
    "가디건": "없음",
    "얇은 긴팔": "없음",
    "긴팔티": "없음",
    "반팔": "없음",
    "민소매": "없음"
}

# =====================================
# ⭐ 추가용품 데이터
# =====================================

cold_accessories = [
    "목도리",
    "장갑",
    "귀마개",
    "핫팩"
]

hot_accessories = [
    "선글라스",
    "모자",
    "쿨토시",
    "손풍기",
    "양산"
]

rain_accessories = [
    "우산",
    "레인부츠"
]

snow_accessories = [
    "목도리",
    "장갑",
    "부츠",
    "핫팩",
    "우산"
]

# =====================================
# ⭐ 추천 함수
# =====================================

def recommend(temp, feel_temp, weather_text, gender):

    # 날씨 숫자 변환
    if weather_text == "맑음":
        weather = 0

    elif weather_text == "비":
        weather = 1

    else:
        weather = 2

    # 머신러닝 예측
    predicted_top = model.predict(
        [[temp, feel_temp, weather]]
    )[0]

    # 성별별 하의 추천
    if gender == "남":
        bottom = random.choice(
            male_bottoms[predicted_top]
        )

    else:
        bottom = random.choice(
            female_bottoms[predicted_top]
        )

    # 이너웨어 추천
    inner = inners[predicted_top]

    # 추가용품 추천
    accessories = []

    # 추울 때
    if feel_temp <= 5:
        accessories.extend(
            random.sample(cold_accessories, 2)
        )

    # 더울 때
    if feel_temp >= 23:
        accessories.extend(
            random.sample(hot_accessories, 2)
        )

    # 비 올 때
    if weather_text == "비":
        accessories.extend(
            random.sample(rain_accessories, 1)
        )

    # 눈 올 때
    if weather_text == "눈":
        accessories.extend(
            random.sample(snow_accessories, 2)
        )

    return f"""
🌡️ 실제 기온: {temp}도
🥶 체감온도: {feel_temp}도
🌦️ 날씨: {weather_text}
👤 성별: {gender}

👕 추천 상의: {predicted_top}
👖 추천 하의: {bottom}
🧣 이너웨어: {inner}

🎒 추가용품:
{', '.join(accessories) if accessories else '없음'}
"""

# =====================================
# ⭐ UI
# =====================================

gr.Interface(
    fn=recommend,

    inputs=[
        gr.Number(label="현재 기온"),
        gr.Number(label="체감온도"),

        gr.Radio(
            ["맑음", "비", "눈"],
            label="날씨 상태"
        ),

        gr.Radio(
            ["남", "여"],
            label="성별"
        )
    ],

    outputs=gr.Textbox(
        label="추천 결과",
        lines=15,
        show_copy_button=True
    ),

    title="🌤️ 머신러닝 기반 옷 추천 시스템"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e912844f8842d0ee65.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


- 우산은 필수로 추천하게 수정
- 옷차림 데이터 세분화